# 🚀 Mon GPT from scratch en C++ et CUDA
Ce notebook génère automatiquement tous les fichiers source, télécharge les données, compile le projet et lance l'entraînement sur le GPU de Colab.

In [ ]:
%%writefile EmbeddingLayer.cu
#include "EmbeddingLayer.cuh"
#include <iostream>
#include <cmath>
#include <cstdlib>

// =========================================================================
// KERNELS CUDA
// =========================================================================



__global__ void embedding_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float beta1 = 0.9f;
        float beta2 = 0.999f;
        float epsilon = 1e-8f;
        m[idx] = beta1 * m[idx] + (1.0f - beta1) * dW[idx];
        v[idx] = beta2 * v[idx] + (1.0f - beta2) * (dW[idx] * dW[idx]);
        float m_hat = m[idx] / (1.0f - powf(beta1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(beta2, (float)t));
        float weight_decay = 0.01f;
        W[idx] -= lr * weight_decay * W[idx];
        W[idx] -= lr * m_hat / (sqrtf(v_hat) + epsilon);
        dW[idx] = 0.0f; 
    }
}


// 1. Kernel Forward : Copie du Dictionnaire + Addition de l'Onde Spatiale (Kernel Fusion)
__global__ void embedding_forward_kernel(int* d_X, float* d_W, float* d_PE, float* d_Y, int embedding_dim, int context_size, int total_elements) {
    int idx = blockDim.x * blockIdx.x + threadIdx.x;
    
    if (idx < total_elements) {
        int embed_idx = idx % embedding_dim;       // Quelle colonne (0 à 255) ?
        int word_pos_global = idx / embedding_dim; // Quel mot dans tout le batch ?
        
        // Position relative de 0 à context_size-1 (pour savoir quelle onde utiliser)
        int seq_pos = word_pos_global % context_size; 
        
        int vocab_id = d_X[word_pos_global];       // L'ID du mot (ex: 45)
        
        // La Fusion : Dictionnaire + Position
        d_Y[idx] = d_W[vocab_id * embedding_dim + embed_idx] + d_PE[seq_pos * embedding_dim + embed_idx];
    }
}

// 2. Kernel Backward : Accumulation des Gradients
__global__ void backward_embedding_kernel(int* d_X, float* d_dY, float* d_dW, int embedding_dim, int total_elements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (idx < total_elements) {
        int embed_idx = idx % embedding_dim;
        int word_pos  = idx / embedding_dim;
        int vocab_id  = d_X[word_pos];
        
        // atomicAdd pour éviter les collisions si un mot apparait plusieurs fois
        atomicAdd(&d_dW[vocab_id * embedding_dim + embed_idx], d_dY[idx]);
    }
}

// 3. Kernel SGD : Mise à jour des poids du Dictionnaire
__global__ void sgd_update_emb(float* param, const float* grad, float lr, int total_size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total_size) {
        param[idx] = param[idx] - (lr * grad[idx]);
    }
}

// =========================================================================
// MÉTHODES DE LA CLASSE
// =========================================================================

EmbeddingLayer::EmbeddingLayer(int vocab_size, int embedding_dim, int batch_size, int context_size) 
    : Layer(batch_size, context_size, embedding_dim) { // Appel au parent !
    
    this->vocab_size = vocab_size;
    this->embedding_dim = embedding_dim;
    this->batch_size = batch_size;
    this->context_size = context_size;

    // 1. Allocations VRAM
    cudaMalloc(&d_W, sizeof(float) * vocab_size * embedding_dim);
    cudaMalloc(&d_dW, sizeof(float) * vocab_size * embedding_dim);
    cudaMalloc(&d_Y, sizeof(float) * batch_size * context_size * embedding_dim);
    cudaMalloc(&d_PE, sizeof(float) * context_size * embedding_dim);

    // 2. Initialisation du Dictionnaire W (CPU -> GPU)
    float *h_W = (float*)malloc(sizeof(float) * vocab_size * embedding_dim);
    for(int i = 0 ; i < vocab_size ; i++) {
        for(int j = 0 ; j < embedding_dim ; j++) {
            h_W[i * embedding_dim + j] = (((float)rand() / RAND_MAX) * 2.0f - 1.0f) * 0.05f;;
        }
    }
    cudaMemcpy(d_W, h_W, sizeof(float) * vocab_size * embedding_dim, cudaMemcpyHostToDevice);
    free(h_W);

    // 3. Calcul de l'Encodage Positionnel (Sinus/Cosinus) sur CPU
    float *h_PE = (float*)malloc(sizeof(float) * context_size * embedding_dim);
    for(int pos = 0; pos < context_size; pos++) {
        for(int i = 0; i < embedding_dim; i+=2) {
            // La formule du papier "Attention Is All You Need"
            float div_term = pow(10000.0f, (float)i / embedding_dim);
            
            // Les dimensions paires reçoivent le sinus
            h_PE[pos * embedding_dim + i] = sin(pos / div_term);
            
            // Les dimensions impaires reçoivent le cosinus
            if(i + 1 < embedding_dim) {
                h_PE[pos * embedding_dim + i + 1] = cos(pos / div_term);
            }
        }
    }
    // Envoi de l'Horloge sur le GPU une bonne fois pour toutes !
    cudaMemcpy(d_PE, h_PE, sizeof(float) * context_size * embedding_dim, cudaMemcpyHostToDevice);
    free(h_PE);
    cudaMalloc(&d_m, sizeof(float) * vocab_size * embedding_dim);
    cudaMalloc(&d_v, sizeof(float) * vocab_size * embedding_dim);
    cudaMemset(d_m, 0, sizeof(float) * vocab_size * embedding_dim);
    cudaMemset(d_v, 0, sizeof(float) * vocab_size * embedding_dim);
}

EmbeddingLayer::~EmbeddingLayer() {
    cudaFree(d_W);
    cudaFree(d_dW);
    cudaFree(d_Y);
    cudaFree(d_PE);
    cudaFree(d_m);
    cudaFree(d_v);
    // Note : On ne free pas d_X car il appartient au DataLoader
}

float* EmbeddingLayer::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    d_X = (int*) d_input; 
    
    // 1. On calcule le nombre total de mots dans tout le batch
    int total_words = batch_size * context_size;
    
    // 2. La grille CUDA s'adapte au nombre de mots (1 thread = 1 mot)
    int threadsPerBlock = 256;
    int blocksPerGrid = (total_words + threadsPerBlock - 1) / threadsPerBlock;
    
    // 3. Appel du kernel avec l'ORDRE EXACT des paramètres
    embedding_forward_kernel<<<blocksPerGrid, threadsPerBlock>>>(
        d_X, d_W, d_PE, d_Y, total_words, context_size, embedding_dim
    );
    
    cudaDeviceSynchronize();
    return d_Y;
}

float* EmbeddingLayer::backward(cublasHandle_t handle, float* d_dY) {
    int total_elements = batch_size * context_size * embedding_dim;
    int threadsPerBlock = 256;
    int blocksPerGrid = (total_elements + threadsPerBlock - 1) / threadsPerBlock;

    // TRÈS IMPORTANT : Remettre le gradient à zéro avant l'accumulation !
    cudaMemset(d_dW, 0, vocab_size * embedding_dim * sizeof(float));

    backward_embedding_kernel<<<blocksPerGrid, threadsPerBlock>>>(
        d_X, d_dY, d_dW, embedding_dim, total_elements
    );
    
    cudaDeviceSynchronize();
    return nullptr; // Le stop absolu.
}

void EmbeddingLayer::step(float learning_rate, int t) {
    int total_elements = vocab_size * embedding_dim;
    int threads = 256;
    int blocks = (total_elements + threads - 1) / threads;
    embedding_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, learning_rate, t, total_elements);
}


In [ ]:
%%writefile LinearLayer.cu
#include "LinearLayer.cuh"
#include <iostream>
__global__ void linear_adam_kernel(float* W, float* dW, float* m, float* v, float lr, int t, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        float beta1 = 0.9f;
        float beta2 = 0.999f;
        float epsilon = 1e-8f;

        m[idx] = beta1 * m[idx] + (1.0f - beta1) * dW[idx];
        v[idx] = beta2 * v[idx] + (1.0f - beta2) * (dW[idx] * dW[idx]);

        float m_hat = m[idx] / (1.0f - powf(beta1, (float)t));
        float v_hat = v[idx] / (1.0f - powf(beta2, (float)t));

        float weight_decay = 0.01f;
        W[idx] -= lr * weight_decay * W[idx];

        W[idx] -= lr * m_hat / (sqrtf(v_hat) + epsilon);
        dW[idx] = 0.0f; // Remise à zéro du gradient !
    }
}
// Kernel pour ajouter les biais
__global__ void add_bias_kernel(float* d_Y, const float* d_b, int batch_features, int out_features) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total_elements = batch_features * out_features;
    if (idx < total_elements) {
        int col = idx % out_features;
        d_Y[idx] += d_b[col];
    }
}

// Kernel pour la descente de gradient
__global__ void sgd_update_linear(float* param, const float* grad, float lr, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        param[idx] -= lr * grad[idx];
    }
}

LinearLayer::LinearLayer(int batch_size, int in_feat, int out_feat) 
    : Layer(batch_size, in_feat, out_feat) {
    
    // Allocations VRAM
    cudaMalloc(&d_W, sizeof(float) * in_feat * out_feat);
    cudaMalloc(&d_b, sizeof(float) * out_feat);
    cudaMalloc(&d_dW, sizeof(float) * in_feat * out_feat);
    cudaMalloc(&d_db, sizeof(float) * out_feat);
    cudaMalloc(&d_Y, sizeof(float) * batch_size * out_feat);
    cudaMalloc(&d_dX, sizeof(float) * batch_size * in_feat);

    // Initialisation aléatoire des poids (CPU -> GPU)
    // Initialisation symétrique (entre -0.05 et +0.05) centrée sur 0 !
    float* h_W = (float*)malloc(sizeof(float) * in_feat * out_feat);
    for(int i = 0; i < in_feat * out_feat; i++) {
        // (rand() / RAND_MAX) donne entre 0 et 1. 
        // * 2.0 - 1.0 donne entre -1 et 1.
        h_W[i] = (((float)rand() / RAND_MAX) * 2.0f - 1.0f) * 0.05f; 
    }
    cudaMemcpy(d_W, h_W, sizeof(float) * in_feat * out_feat, cudaMemcpyHostToDevice);
    
    cudaMemset(d_b, 0, sizeof(float) * out_feat); // Biais à zéro
    
    cudaMalloc(&d_m, sizeof(float) * in_feat * out_feat);
    cudaMalloc(&d_v, sizeof(float) * in_feat * out_feat);
    cudaMemset(d_m, 0, sizeof(float) * in_feat * out_feat);
    cudaMemset(d_v, 0, sizeof(float) * in_feat * out_feat);

    free(h_W);
}

LinearLayer::~LinearLayer() {
    cudaFree(d_W); cudaFree(d_b); cudaFree(d_dW);
    cudaFree(d_db); cudaFree(d_Y); cudaFree(d_dX);
    cudaFree(d_m);
    cudaFree(d_v);
}

float* LinearLayer::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    d_X_cache = (float*)d_input; // On sauvegarde l'entrée pour le backward !
    
    const float alpha = 1.0f;
    const float beta = 0.0f;
    
    // d_Y = d_X * d_W
    // Rappel cuBLAS (Column-Major) : on calcule W^T * X^T pour obtenir Y^T
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                out_features, batch_features, in_features,
                &alpha,
                d_W, out_features,
                d_X_cache, in_features,
                &beta,
                d_Y, out_features);

    // Ajout des biais
    int threads = 256;
    int blocks = (batch_features * out_features + threads - 1) / threads;
    add_bias_kernel<<<blocks, threads>>>(d_Y, d_b, batch_features, out_features);
    
    // (Dans un code complet, on appliquerait le ReLU ici si activation_type == ACTIVATION_RELU)

    return d_Y;
}

float* LinearLayer::backward(cublasHandle_t handle, float* d_dY) {
    const float alpha = 1.0f;
    const float beta = 0.0f;

    // 1. Calcul de dX = dY * W^T (Pour la couche d'en dessous)
    cublasSgemm(handle, CUBLAS_OP_T, CUBLAS_OP_N,
                in_features, batch_features, out_features,
                &alpha, d_W, out_features, d_dY, out_features,
                &beta, d_dX, in_features);

    // 2. Calcul de dW = X^T * dY (Pour mettre à jour nos propres poids)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_T,
                out_features, in_features, batch_features,
                &alpha, d_dY, out_features, d_X_cache, in_features,
                &beta, d_dW, out_features);
    return d_dX;
}

void LinearLayer::step(float learning_rate, int t) {
    int total_elements = in_features * out_features;
    int threads = 256;
    int blocks = (total_elements + threads - 1) / threads;
    linear_adam_kernel<<<blocks, threads>>>(d_W, d_dW, d_m, d_v, learning_rate, t, total_elements);
}


In [ ]:
%%writefile TransformerBlock.cu
#include "TransformerBlock.cuh"

// Le Kernel qui sauve l'IA : Addition élément par élément (C = A + B)
__global__ void add_tensors_kernel(float* A, float* B, float* C, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        C[idx] = A[idx] + B[idx];
    }
}

TransformerBlock::TransformerBlock(int batch_size, int context_size, int embedding_dim)
    : Layer(batch_size, context_size, embedding_dim) {
    
    this->batch_size = batch_size;
    this->context_size = context_size;
    this->embedding_dim = embedding_dim;

    norm1 = new RMSNormLayer(batch_size, context_size, embedding_dim);
    attn  = new AttentionLayer(batch_size, context_size, embedding_dim);
    norm2 = new RMSNormLayer(batch_size, context_size, embedding_dim);
    ffn   = new FeedForwardLayer(batch_size, context_size, embedding_dim, 4);

    int total_elements = batch_size * context_size * embedding_dim;
    
    // Allocation pour les câbles de contournement
    cudaMalloc(&d_res1, sizeof(float) * total_elements);
    cudaMalloc(&d_out,  sizeof(float) * total_elements);
    cudaMalloc(&d_dRes1,sizeof(float) * total_elements);
    cudaMalloc(&d_dX,   sizeof(float) * total_elements);
}

TransformerBlock::~TransformerBlock() {
    delete norm1; delete attn; delete norm2; delete ffn;
    cudaFree(d_res1); cudaFree(d_out);
    cudaFree(d_dRes1); cudaFree(d_dX);
}

float* TransformerBlock::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    float* d_X = (float*)d_input;
    int total = batch_size * context_size * embedding_dim;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;

    // --- CHEMIN 1 : L'ATTENTION ---
    float* d_norm1 = norm1->forward(handle, d_X, ACTIVATION_NONE);
    float* d_attn  = attn->forward(handle, d_norm1, ACTIVATION_NONE);
    // CABLE DE SAUVETAGE 1 : On additionne l'entrée et la sortie !
    add_tensors_kernel<<<blocks, threads>>>(d_X, d_attn, d_res1, total);
    cudaDeviceSynchronize();

    // --- CHEMIN 2 : LE FEEDFORWARD ---
    float* d_norm2 = norm2->forward(handle, d_res1, ACTIVATION_NONE);
    float* d_ffn   = ffn->forward(handle, d_norm2, ACTIVATION_NONE);
    // CABLE DE SAUVETAGE 2 : On additionne le chemin 1 et le FFN !
    add_tensors_kernel<<<blocks, threads>>>(d_res1, d_ffn, d_out, total);
    cudaDeviceSynchronize();

    return d_out;
}

float* TransformerBlock::backward(cublasHandle_t handle, float* d_dY) {
    int total = batch_size * context_size * embedding_dim;
    int threads = 256;
    int blocks = (total + threads - 1) / threads;

    // --- RETOUR CHEMIN 2 (FFN) ---
    float* d_dFFN = ffn->backward(handle, d_dY);
    float* d_dNorm2_out = norm2->backward(handle, d_dFFN);
    // Jonction du câble : On additionne le gradient direct (d_dY) et le gradient du FFN
    add_tensors_kernel<<<blocks, threads>>>(d_dY, d_dNorm2_out, d_dRes1, total);
    cudaDeviceSynchronize();

    // --- RETOUR CHEMIN 1 (Attention) ---
    float* d_dAttn = attn->backward(handle, d_dRes1);
    float* d_dNorm1_out = norm1->backward(handle, d_dAttn);
    // Jonction du câble final : On additionne le gradient du chemin 2 et de l'Attention
    add_tensors_kernel<<<blocks, threads>>>(d_dRes1, d_dNorm1_out, d_dX, total);
    cudaDeviceSynchronize();

    return d_dX;
}

void TransformerBlock::step(float learning_rate, int t) {
    norm1->step(learning_rate, t);
    attn->step(learning_rate, t);
    norm2->step(learning_rate, t);
    ffn->step(learning_rate, t);
}


In [ ]:
%%writefile FeedForwardLayer.cu
#include "FeedForwardLayer.cuh"
#include <iostream>

// =========================================================================
// MÉTHODES DE LA CLASSE
// =========================================================================

FeedForwardLayer::FeedForwardLayer(int batch_size, int context_size, int embedding_dim, int expansion_factor) 
    : Layer(batch_size, context_size, embedding_dim) {
    
    this->batch_size = batch_size;
    this->context_size = context_size;
    this->embedding_dim = embedding_dim;
    
    // Le secret du Transformer : La couche cachée est 4 fois plus grande !
    this->hidden_dim = embedding_dim * expansion_factor;

    // Astuce classique : on aplatit le temps et le batch
    int flat_batch = batch_size * context_size;

    // Instanciation de nos deux couches
    // Couche 1 : Entrée = embedding_dim, Sortie = hidden_dim (ex: 256 -> 1024)
    fc1 = new LinearLayer(flat_batch, embedding_dim, hidden_dim);
    
    // Couche 2 : Entrée = hidden_dim, Sortie = embedding_dim (ex: 1024 -> 256)
    fc2 = new LinearLayer(flat_batch, hidden_dim, embedding_dim);
}

FeedForwardLayer::~FeedForwardLayer() {
    delete fc1;
    delete fc2;
}

float* FeedForwardLayer::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    // Étape 1 : Expansion avec activation (Le ReLU coupe les valeurs négatives)
    // Note : On utilise l'activation DANS la première couche
    float* d_hidden = fc1->forward(handle, (float*)d_input, ACTIVATION_RELU);

    // Étape 2 : Contraction vers la taille d'origine (Pas d'activation ici !)
    float* d_output = fc2->forward(handle, d_hidden, ACTIVATION_NONE);

    return d_output;
}

float* FeedForwardLayer::backward(cublasHandle_t handle, float* d_dY) {
    // La rétropropagation est magique grâce à notre architecture objet :
    // 1. Le gradient traverse la couche 2 (qui nous renvoie le gradient intermédiaire)
    float* d_dHidden = fc2->backward(handle, d_dY);

    // 2. Ce gradient intermédiaire traverse la couche 1 (qui nous renvoie le dX final)
    float* d_dX = fc1->backward(handle, d_dHidden);
    
    // (Dans un code ultra-complet, on stockerait d_dX dans la classe pour le retourner
    // à la couche d'Attention qui se trouve en dessous)
    return d_dX;
}

void FeedForwardLayer::step(float learning_rate, int t) {
    fc1->step(learning_rate, t);
    fc2->step(learning_rate, t);
}


In [ ]:
%%writefile main.cu
#include <iostream>
#include <cublas_v2.h>
#include "GPTModel.cuh"
#include "DataLoader.h"
#include <vector>
#include <string>

__global__ void check_nan_kernel(float* tensor, int size, const char* nom_couche) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        if (isnan(tensor[idx]) || isinf(tensor[idx])) {
            printf("🚨 ALERTE : NaN ou Inf detecte dans la couche : %s (Index %d)\n", nom_couche, idx);
        }
    }
}

// Kernel de sécurité : Empêche les gradients d'exploser (Gradient Clipping)
__global__ void clip_gradients_kernel(float* d_dY, float min_val, float max_val, int total_elements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total_elements) {
        float val = d_dY[idx];
        if (val > max_val) val = max_val;
        if (val < min_val) val = min_val;
        // La protection suprême contre les NaN générés plus haut :
        if (isnan(val)) val = 0.0f; 
        d_dY[idx] = val;
    }
}
// Fonction CPU pour générer du texte avec le modèle entraîné
// --- NOUVELLE FONCTION GENERATE_TEXT ---
void generate_text(cublasHandle_t handle, GPTModel* model, std::string prompt, int length_to_generate, char* int_to_char, int context_size, int vocab_size) {
    std::cout << "\nAmorce : \"" << prompt << "\"" << std::endl;
    std::cout << "Résultat : " << prompt;

    std::vector<int> current_context;
    
    // 1. CORRECTION : On utilise le VRAI dictionnaire pour traduire le prompt !
    for (char c : prompt) {
        int token = 0; // Token par défaut si caractère inconnu
        for(int v = 0; v < vocab_size; v++) {
            if (int_to_char[v] == c) {
                token = v;
                break;
            }
        }
        current_context.push_back(token);
    }

    int* d_X_gen;
    cudaMalloc(&d_X_gen, sizeof(int) * context_size);
    float* h_logits = (float*)malloc(sizeof(float) * context_size * vocab_size);

    for (int i = 0; i < length_to_generate; i++) {
        std::vector<int> input_window;
        int start_idx = std::max(0, (int)current_context.size() - context_size);
        for (int j = start_idx; j < current_context.size(); j++) {
            input_window.push_back(current_context[j]);
        }
        while(input_window.size() < context_size) input_window.push_back(0); 

        cudaMemcpy(d_X_gen, input_window.data(), sizeof(int) * context_size, cudaMemcpyHostToDevice);

        // Température ajoutée implicitement en ne modifiant pas les logits bruts
        float* d_logits_out = model->forward(handle, d_X_gen, 0);
        cudaMemcpy(h_logits, d_logits_out, sizeof(float) * context_size * vocab_size, cudaMemcpyDeviceToHost);
        
        int last_word_offset = (context_size - 1) * vocab_size;

        float max_l = -1e9f;
        for(int v=0; v<vocab_size; v++) max_l = std::max(max_l, h_logits[last_word_offset + v]);
        
        float sum_exp = 0.0f;
        std::vector<float> probs(vocab_size);
        for(int v=0; v<vocab_size; v++) {
            probs[v] = expf(h_logits[last_word_offset + v] - max_l);
            sum_exp += probs[v];
        }
        
        float r = ((float)rand() / RAND_MAX) * sum_exp;
        float cumulative = 0.0f;
        int next_token = 0;
        
        for(int v=0; v<vocab_size; v++) {
            cumulative += probs[v];
            if (r <= cumulative) {
                next_token = v;
                break;
            }
        }

        std::cout << int_to_char[next_token] << std::flush; // On affiche instantanément
        current_context.push_back(next_token);
    }
    
    std::cout << std::endl;
    cudaFree(d_X_gen);
    free(h_logits);
}

// Kernel magique : Fused Softmax + Cross Entropy Backward
// Il transforme les Logits bruts en gradients d_dY prêts à être rétropropagés.
__global__ void cross_entropy_backward_kernel(float* d_logits, int* d_targets, float* d_dY, int vocab_size, int total_words) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Un thread = un mot du batch
    
    if (idx < total_words) {
        int target_class = d_targets[idx]; // Le vrai mot attendu
        
        // 1. Softmax local (très simplifié ici pour l'exemple)
        float max_val = -1e9f;
        for (int i = 0; i < vocab_size; i++) {
            max_val = fmaxf(max_val, d_logits[idx * vocab_size + i]);
        }
        
        float sum_exp = 0.0f;
        for (int i = 0; i < vocab_size; i++) {
            sum_exp += expf(d_logits[idx * vocab_size + i] - max_val);
        }
        
        // 2. Calcul du gradient dY = (Probas - Cible) / total_words
        for (int i = 0; i < vocab_size; i++) {
            float prob = expf(d_logits[idx * vocab_size + i] - max_val) / (sum_exp + 1e-7f);
            
            if (i == target_class) {
                d_dY[idx * vocab_size + i] = (prob - 1.0f) / total_words; // <-- AJOUT DE LA DIVISION
            } else {
                d_dY[idx * vocab_size + i] = (prob - 0.0f) / total_words; // <-- AJOUT DE LA DIVISION
            }
        }
    }
}

int main() {
    cublasHandle_t handle;
    cublasCreate(&handle);

    // 1. HYPERPARAMÈTRES INITIAUX
    int context_size = 32;
    int batch_size = 128;
    int embedding_dim = 128;
    int num_blocks = 4;
    float learning_rate = 3e-4f;
    int iterations = 10000;

    std::cout << "--- CHARGEMENT DES DONNEES ---" << std::endl;
    // Assure-toi d'avoir un fichier "input.txt" dans le même dossier !
    DataLoader dataloader("input.txt", batch_size, context_size);
    
    // Le DataLoader décide du vocab_size réel !
    int vocab_size = dataloader.get_vocab_size(); 

    std::cout << "--- CREATION DU MODELE GPT ---" << std::endl;
    GPTModel model(vocab_size, embedding_dim, batch_size, context_size, num_blocks);
    
    int total_words = batch_size * context_size;
    
    // 2. ALLOCATIONS MÉMOIRE
    // Mémoire CPU (Host)
    int* h_X = (int*)malloc(sizeof(int) * total_words);
    int* h_targets = (int*)malloc(sizeof(int) * total_words);

    // Mémoire GPU (Device)
    int* d_X;       
    int* d_targets; 
    float* d_dY;    
    cudaMalloc(&d_X, sizeof(int) * total_words);
    cudaMalloc(&d_targets, sizeof(int) * total_words);
    cudaMalloc(&d_dY, sizeof(float) * total_words * vocab_size);

    std::cout << "--- DEBUT DE L'ENTRAINEMENT ---" << std::endl;

    for (int iter = 0; iter < iterations; iter++) {
        
        dataloader.get_batch(h_X, h_targets);
        
        cudaMemcpy(d_X, h_X, sizeof(int) * total_words, cudaMemcpyHostToDevice);
        cudaMemcpy(d_targets, h_targets, sizeof(int) * total_words, cudaMemcpyHostToDevice);
        
        float* d_logits = model.forward(handle, d_X, 0);
        // Radar à NaN :
        int total_logits = total_words * vocab_size;
        check_nan_kernel<<<(total_logits + 255)/256, 256>>>(d_logits, total_logits, "SORTIE_LOGITS");
        cudaDeviceSynchronize();

        // NOUVEAU : Affichage de la Loss tous les 100 pas (Calcul sur CPU)
        if (iter % 100 == 0) {
            float* h_logits = (float*)malloc(sizeof(float) * total_words * vocab_size);
            cudaMemcpy(h_logits, d_logits, sizeof(float) * total_words * vocab_size, cudaMemcpyDeviceToHost);
            
            float loss = 0.0f;
            for(int i = 0; i < total_words; i++) {
                int target = h_targets[i];
                float max_l = -1e9f;
                for(int v=0; v<vocab_size; v++) max_l = std::max(max_l, h_logits[i*vocab_size + v]);
                
                float sum_exp = 0.0f;
                for(int v=0; v<vocab_size; v++) sum_exp += expf(h_logits[i*vocab_size + v] - max_l);
                
                float prob = expf(h_logits[i*vocab_size + target] - max_l) / sum_exp;
                loss += -logf(prob + 1e-7f); // Formule mathématique de la Cross-Entropy
            }
            loss /= total_words;
            std::cout << "Iteration " << iter << " / " << iterations << " | Loss: " << loss << std::endl;
            free(h_logits);
        }

        int threads = 256;
        int blocks = (total_words + threads - 1) / threads;
        cross_entropy_backward_kernel<<<blocks, threads>>>(d_logits, d_targets, d_dY, vocab_size, total_words);
        cudaDeviceSynchronize();

        // NOUVEAU : Le bouclier anti-explosion ! On limite l'erreur entre -1.0 et 1.0
        int total_grad_elements = total_words * vocab_size;
        int blocks_clip = (total_grad_elements + threads - 1) / threads;
        clip_gradients_kernel<<<blocks_clip, threads>>>(d_dY, -1.0f, 1.0f, total_grad_elements);
        cudaDeviceSynchronize();

        

        model.backward(handle, d_dY);
        model.step(learning_rate, iter + 1);
    }

    std::cout << "--- ENTRAINEMENT TERMINE ---" << std::endl;

    // --- TEST DE GÉNÉRATION ---
    // On extrait le dictionnaire pour la fonction generate_text
    std::map<int, char> int_to_char_map = dataloader.get_int_to_char_map();
    char* int_to_char_array = (char*)malloc(sizeof(char) * vocab_size);
    for(int i=0; i<vocab_size; i++) int_to_char_array[i] = int_to_char_map[i];

    std::cout << "\n--- GENERATION DE TEXTE ---" << std::endl;
    generate_text(handle, &model, "The city", 50, int_to_char_array, context_size, vocab_size);
    generate_text(handle, &model, "Romeo, ", 50, int_to_char_array, context_size, vocab_size);

    // NETTOYAGE
    free(h_X); free(h_targets); free(int_to_char_array);
    cudaFree(d_X); cudaFree(d_targets); cudaFree(d_dY);
    cublasDestroy(handle);

    return 0;
}


In [ ]:
%%writefile GPTModel.cu
#include "GPTModel.cuh"
#include <iostream>

// =========================================================================
// MÉTHODES DE LA CLASSE
// =========================================================================

GPTModel::GPTModel(int vocab_size, int embedding_dim, int batch_size, int context_size, int num_blocks) 
    : Layer(batch_size, context_size, embedding_dim) {
    
    this->vocab_size = vocab_size;
    this->embedding_dim = embedding_dim;
    this->batch_size = batch_size;
    this->context_size = context_size;
    this->num_blocks = num_blocks;

    // 1. Instanciation de l'Embedding
    embedding = new EmbeddingLayer(vocab_size, embedding_dim, batch_size, context_size);

    // 2. Instanciation de la tour de Blocs Transformer
    for (int i = 0; i < num_blocks; i++) {
        blocks.push_back(new TransformerBlock(batch_size, context_size, embedding_dim));
    }

    // 3. Instanciation de la RMSNorm finale
    final_norm = new RMSNormLayer(batch_size, context_size, embedding_dim);

    // 4. Instanciation de la LM Head
    // Entrée : Les 256 floats de la pensée finale du Transformer
    // Sortie : Les 65 floats de probabilité pour chaque lettre du vocabulaire
    int flat_batch = batch_size * context_size;
    lm_head = new LinearLayer(flat_batch, embedding_dim, vocab_size);
}

GPTModel::~GPTModel() {
    delete embedding;
    for (int i = 0; i < num_blocks; i++) {
        delete blocks[i];
    }
    delete final_norm;
    delete lm_head;
}

float* GPTModel::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    // 1. On entre dans l'Embedding (d_input est notre int* du DataLoader)
    float* d_out = embedding->forward(handle, d_input, ACTIVATION_NONE);

    // 2. On traverse la tour de Blocs Transformer
    for (int i = 0; i < num_blocks; i++) {
        // La sortie du bloc i devient l'entrée du bloc i+1 !
        d_out = blocks[i]->forward(handle, d_out, ACTIVATION_NONE);
    }

    // 3. On stabilise une dernière fois
    d_out = final_norm->forward(handle, d_out, ACTIVATION_NONE);

    // 4. On projette vers le vocabulaire
    // d_out contient maintenant nos Logits ! (Dimension: batch_size * context_size * vocab_size)
    float* d_logits = lm_head->forward(handle, d_out, ACTIVATION_NONE);
    return d_logits;
}

float* GPTModel::backward(cublasHandle_t handle, float* d_dY) {
    // d_dY est l'erreur calculée par la fonction de perte (Cross-Entropy).
    
    // 1. Rétropropagation dans la tête de prédiction
    float* d_grad = lm_head->backward(handle, d_dY);

    // 2. Rétropropagation dans la norme finale
    d_grad = final_norm->backward(handle, d_grad);

    // 3. Rétropropagation dans la tour de Blocs (À L'ENVERS !)
    // On part du dernier bloc (num_blocks - 1) jusqu'au premier (0)
    for (int i = num_blocks - 1; i >= 0; i--) {
        d_grad = blocks[i]->backward(handle, d_grad);
    }

    // 4. Rétropropagation finale dans l'Embedding (qui met à jour le dictionnaire)
    embedding->backward(handle, d_grad);
    return nullptr; // Fin de la boucle pour le modèle
}

void GPTModel::step(float learning_rate, int t) {
    embedding->step(learning_rate, t);
    for (int i = 0; i < num_blocks; i++) {
        blocks[i]->step(learning_rate, t);
    }
    final_norm->step(learning_rate, t);
    lm_head->step(learning_rate, t);
}


In [ ]:
%%writefile AttentionLayer.cu
#include "AttentionLayer.cuh"
#include <iostream>
#include <cmath>

// =========================================================================
// KERNELS CUDA (Les petits outils de l'Attention)
// =========================================================================
// Kernel CUDA pour appliquer le Masque Causal (empêcher de tricher)
__global__ void causal_mask_kernel(float* d_Scores, int context_size, int batch_size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    int total_elements = batch_size * context_size * context_size;
    
    if (idx < total_elements) {
        // On retrouve notre position dans la matrice 2D [context_size x context_size]
        int matrix_idx = idx % (context_size * context_size);
        int row = matrix_idx / context_size; // Le mot qui "regarde"
        int col = matrix_idx % context_size; // Le mot qui "est regardé"
        
        // Si on essaie de regarder dans le futur (colonne strictement supérieure à ligne)
        if (col > row) {
            d_Scores[idx] = -1e9f; // -Infini pour tuer l'attention vers le futur
        }
    }
}
// 1. Kernel pour diviser les scores par la racine carrée de la dimension
__global__ void scale_scores_kernel(float* d_Scores, float scale_factor, int total_elements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total_elements) {
        d_Scores[idx] = d_Scores[idx] * scale_factor;
    }
}
// Kernel CUDA pour un Softmax ultra-robuste (Forward)
// Kernel CUDA pour un Softmax ultra-robuste et SÉCURISÉ (Forward)
__global__ void softmax_forward_kernel(float* d_Scores, int context_size, int batch_size) {
    int row = blockIdx.x; 
    int tid = threadIdx.x; 
    
    // NOUVEAU : Mémoire partagée pour éviter que les threads se marchent dessus
    extern __shared__ float shared_exp[]; 
    
    if (row < batch_size * context_size && tid < context_size) {
        int base_idx = row * context_size;
        
        // 1. Recherche du Max (Rapide)
        float max_val = -1e9f;
        for (int i = 0; i < context_size; i++) {
            max_val = fmaxf(max_val, d_Scores[base_idx + i]);
        }
        
        // 2. Calcul de l'exponentielle stocké en lieu sûr !
        float my_exp = expf(d_Scores[base_idx + tid] - max_val);
        shared_exp[tid] = my_exp;
        
        // BARRIÈRE DE SÉCURITÉ : On attend que tout le monde ait posé son calcul
        __syncthreads(); 
        
        // 3. Calcul de la somme à partir de la mémoire partagée (protégée)
        float sum_exp = 0.0f;
        for (int i = 0; i < context_size; i++) {
            sum_exp += shared_exp[i];
        }
        
        // 4. Écriture finale (aucun risque de conflit)
        d_Scores[base_idx + tid] = my_exp / (sum_exp + 1e-9f);
    }
}
// 2. Kernel Softmax (Par ligne). 
// Chaque bloc gère UNE ligne de la matrice des scores (un mot qui regarde les autres)
__global__ void softmax_attention_kernel(float* d_Scores, int context_size) {
    int row_idx = blockIdx.x; // Quelle ligne (quel mot) on traite ?
    int col_idx = threadIdx.x; // Quelle colonne (quel mot on regarde) ?
    
    extern __shared__ float shared_scores[];

    if (col_idx < context_size) {
        int global_idx = row_idx * context_size + col_idx;
        float val = d_Scores[global_idx];
        
        // --- 1. Recherche du Max pour la stabilité numérique ---
        shared_scores[col_idx] = val;
        __syncthreads();

        for (int stride = 1; stride < blockDim.x; stride *= 2) {
            int index = 2 * stride * col_idx;
            if (index + stride < blockDim.x) {
                shared_scores[index] = fmaxf(shared_scores[index], shared_scores[index + stride]);
            }
            __syncthreads();
        }
        float max_val = shared_scores[0];
        __syncthreads();

        // --- 2. Exponentielle (e^x) ---
        val = expf(val - max_val);
        shared_scores[col_idx] = val;
        __syncthreads();

        // --- 3. Somme de la ligne ---
        for (int stride = 1; stride < blockDim.x; stride *= 2) {
            int index = 2 * stride * col_idx;
            if (index + stride < blockDim.x) {
                shared_scores[index] += shared_scores[index + stride];
            }
            __syncthreads();
        }
        float sum = shared_scores[0];
        __syncthreads();

        // --- 4. Division finale (Pourcentage) ---
        d_Scores[global_idx] = val / (sum + 1e-9f);
    }
}


// Kernel CUDA pour la dérivée du Softmax (Backward)
__global__ void softmax_backward_kernel(float* d_dScores_softmax, float* d_Scores_softmax, float* d_dScores, int context_size) {
    int row_idx = blockIdx.x; 
    int col_idx = threadIdx.x;

    if (col_idx < context_size) {
        int global_idx = row_idx * context_size + col_idx;
        
        // 1. Calculer la somme (dScores_softmax * Scores_softmax) pour cette ligne
        float local_dot = d_dScores_softmax[global_idx] * d_Scores_softmax[global_idx];
        
        // (Pour faire simple sans mémoire partagée complète ici, 
        // on suppose une réduction classique ou une boucle sur la ligne)
        float row_sum = 0.0f;
        for(int i = 0; i < context_size; i++) {
            int i_idx = row_idx * context_size + i;
            row_sum += d_dScores_softmax[i_idx] * d_Scores_softmax[i_idx];
        }

        // 2. Appliquer la formule de la dérivée du Softmax
        d_dScores[global_idx] = d_Scores_softmax[global_idx] * (d_dScores_softmax[global_idx] - row_sum);
    }
}

// Kernel CUDA pour additionner les 3 gradients d'entrée
__global__ void sum_gradients_kernel(float* d_dX, const float* d_dX_q, const float* d_dX_k, const float* d_dX_v, int total_elements) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    
    // Sécurité classique
    if (idx < total_elements) {
        d_dX[idx] = d_dX_q[idx] + d_dX_k[idx] + d_dX_v[idx];
    }
}
// =========================================================================
// MÉTHODES DE LA CLASSE
// =========================================================================

AttentionLayer::AttentionLayer(int batch_size, int context_size, int embedding_dim) 
    : Layer(batch_size, context_size, embedding_dim) {
    
    this->batch_size = batch_size;
    this->context_size = context_size;
    this->embedding_dim = embedding_dim;

    // 1. Création de nos sous-couches LinearLayer.
    // L'astuce : on aplatit le batch_size et le context_size pour nos LinearLayers.
    int flat_batch = batch_size * context_size;
    
    W_q = new LinearLayer(flat_batch, embedding_dim, embedding_dim);
    W_k = new LinearLayer(flat_batch, embedding_dim, embedding_dim);
    W_v = new LinearLayer(flat_batch, embedding_dim, embedding_dim);
    W_o = new LinearLayer(flat_batch, embedding_dim, embedding_dim);

    // 2. Allocation des espaces mémoires de l'Attention
    // La matrice de Scores a pour taille [batch_size, context_size, context_size]
    cudaMalloc(&d_Scores, sizeof(float) * batch_size * context_size * context_size);
    
    // Le résultat (Scores * V) a pour taille [batch_size, context_size, embedding_dim]
    cudaMalloc(&d_AttentionOut, sizeof(float) * batch_size * context_size * embedding_dim);
    // NOUVEAU : Allocation VRAM pour la passe Backward
    int total_elements = batch_size * context_size * embedding_dim;
    int total_scores = batch_size * context_size * context_size;

    cudaMalloc(&d_Q_grad, sizeof(float) * total_elements);
    cudaMalloc(&d_K_grad, sizeof(float) * total_elements);
    cudaMalloc(&d_V_grad, sizeof(float) * total_elements);
    cudaMalloc(&d_dX, sizeof(float) * total_elements);

    cudaMalloc(&d_dScores_softmax, sizeof(float) * total_scores);
    cudaMalloc(&d_dScores, sizeof(float) * total_scores);
}

AttentionLayer::~AttentionLayer() {
    delete W_q;
    delete W_k;
    delete W_v;
    delete W_o;
    cudaFree(d_Scores);
    cudaFree(d_AttentionOut);
    // NOUVEAU : Nettoyage du Backward
    cudaFree(d_Q_grad);
    cudaFree(d_K_grad);
    cudaFree(d_V_grad);
    cudaFree(d_dX);
    cudaFree(d_dScores_softmax);
    cudaFree(d_dScores);
}

float* AttentionLayer::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    float* d_X = (float*) d_input;
    const float alpha = 1.0f;
    const float beta = 0.0f;

    // =====================================================================
    // ÉTAPE 1 : Générer les matrices Q, K et V
    // =====================================================================
    // Nos LinearLayer gèrent déjà leurs propres allocations internes pour la sortie (d_Y).
    // On récupère juste les pointeurs !
    d_Q = W_q->forward(handle, d_X, ACTIVATION_NONE);
    d_K = W_k->forward(handle, d_X, ACTIVATION_NONE);
    d_V = W_v->forward(handle, d_X, ACTIVATION_NONE);

    // =====================================================================
    // ÉTAPE 2 : Produit Scalaire (Scores = Q * K^T)
    // =====================================================================
    // On utilise la fonction Batched pour ne pas mélanger les phrases du batch !
    // ATTENTION PIÈGE cuBLAS : cuBLAS lit en Column-Major. 
    // Pour calculer (Q * K^T) en Row-Major C++, on demande à cuBLAS de calculer (K^T * Q).
    
    long long int stride_Q = context_size * embedding_dim;
    long long int stride_K = context_size * embedding_dim;
    long long int stride_Scores = context_size * context_size;

    cublasSgemmStridedBatched(
        handle,
        CUBLAS_OP_T, CUBLAS_OP_N, // Transposer K, Ne pas transposer Q
        context_size, context_size, embedding_dim, // m, n, k
        &alpha,
        d_K, embedding_dim, stride_K, // Matrice A (qui est K)
        d_Q, embedding_dim, stride_Q, // Matrice B (qui est Q)
        &beta,
        d_Scores, context_size, stride_Scores, // Matrice C (le résultat)
        batch_size // Le nombre de phrases indépendantes
    );

    // =====================================================================
    // ÉTAPE 3 : Mise à l'échelle (Scale)
    // =====================================================================
    int total_scores = batch_size * context_size * context_size;
    int threads_scale = 256;
    int blocks_scale = (total_scores + threads_scale - 1) / threads_scale;
    float scale_factor = 1.0f / sqrtf((float)embedding_dim); // 1.0 divisé par la racine !
    // Appel du kernel pour multiplier d_Scores par scale_factor
    
    scale_scores_kernel<<<blocks_scale, threads_scale>>>(d_Scores, scale_factor, total_scores);
    cudaDeviceSynchronize();

    // On applique le masque
    int threads_mask = 256;
    int blocks_mask = (total_scores + threads_mask - 1) / threads_mask;
    
    causal_mask_kernel<<<blocks_mask, threads_mask>>>(d_Scores, context_size, batch_size);
    cudaDeviceSynchronize();

    // =====================================================================
    // ÉTAPE 4 : Softmax (Les Pourcentages d'Attention)
    // =====================================================================
    int blocks_softmax = batch_size * context_size; // Une ligne = un bloc
    int threads_softmax = context_size; // Un thread par colonne (mot)
    size_t shared_mem = context_size * sizeof(float);
    
    softmax_attention_kernel<<<blocks_softmax, threads_softmax, shared_mem>>>(d_Scores, context_size,batch_size);
    cudaDeviceSynchronize();

    // =====================================================================
    // ÉTAPE 5 : Le Mélange (Output = Scores * V)
    // =====================================================================
    // Encore une inversion cuBLAS : (Scores * V) en Row-Major = (V * Scores) en Col-Major.
    long long int stride_V = context_size * embedding_dim;
    long long int stride_Out = context_size * embedding_dim;

    cublasSgemmStridedBatched(
        handle,
        CUBLAS_OP_N, CUBLAS_OP_N,
        embedding_dim, context_size, context_size, // m, n, k
        &alpha,
        d_V, embedding_dim, stride_V, // Matrice A (qui est V)
        d_Scores, context_size, stride_Scores, // Matrice B (qui est Scores)
        &beta,
        d_AttentionOut, embedding_dim, stride_Out, // Matrice C (Résultat)
        batch_size
    );

    // =====================================================================
    // ÉTAPE 6 : Projection finale (W_o)
    // =====================================================================
    return W_o->forward(handle, d_AttentionOut, ACTIVATION_NONE);
}

float* AttentionLayer::backward(cublasHandle_t handle, float* d_dY) {
    const float alpha = 1.0f;
    const float beta = 0.0f;
    long long int stride_Q = context_size * embedding_dim;
    long long int stride_Scores = context_size * context_size;

    // 1. Backward de W_o
    // (Suppose que tes LinearLayer::backward retournent float* d_dX)
    float* d_dOut = W_o->backward(handle, d_dY); 

    // 2. Backward de Output = Scores * V 
    // dV = Scores^T * dOut
    cublasSgemmStridedBatched(handle, CUBLAS_OP_N, CUBLAS_OP_T,
        embedding_dim, context_size, context_size, &alpha,
        d_dOut, embedding_dim, stride_Q,
        d_Scores, context_size, stride_Scores, &beta,
        d_V_grad, embedding_dim, stride_Q, batch_size); // Il faut un pointeur d_V_grad alloué !

    // dScores_softmax = dOut * V^T
    cublasSgemmStridedBatched(handle, CUBLAS_OP_T, CUBLAS_OP_N,
        context_size, context_size, embedding_dim, &alpha,
        d_V, embedding_dim, stride_Q,
        d_dOut, embedding_dim, stride_Q, &beta,
        d_dScores_softmax, context_size, stride_Scores, batch_size);

    // 3. Backward du Softmax
    int blocks_softmax = batch_size * context_size;
    int threads_softmax = context_size;
    softmax_backward_kernel<<<blocks_softmax, threads_softmax>>>(d_dScores_softmax, d_Scores, d_dScores, context_size);
    cudaDeviceSynchronize();

    // 4. Backward du Scale (Mise à l'échelle)
    float scale_factor = 1.0f / sqrtf((float)embedding_dim);
    int total_scores = batch_size * context_size * context_size;
    int threads_scale = 256;
    int blocks_scale = (total_scores + threads_scale - 1) / threads_scale;
    // On réutilise scale_scores_kernel car c'est la même division mathématique !
    scale_scores_kernel<<<blocks_scale, threads_scale>>>(d_dScores, scale_factor, total_scores); 

    // 5. Backward de Scores = Q * K^T
    // dQ = dScores * K
    cublasSgemmStridedBatched(handle, CUBLAS_OP_N, CUBLAS_OP_N,
        embedding_dim, context_size, context_size, &alpha,
        d_K, embedding_dim, stride_Q,
        d_dScores, context_size, stride_Scores, &beta,
        d_Q_grad, embedding_dim, stride_Q, batch_size);

    // dK = dScores^T * Q
    cublasSgemmStridedBatched(handle, CUBLAS_OP_N, CUBLAS_OP_T,
        embedding_dim, context_size, context_size, &alpha,
        d_Q, embedding_dim, stride_Q,
        d_dScores, context_size, stride_Scores, &beta,
        d_K_grad, embedding_dim, stride_Q, batch_size);

    // 6. Backward des projections initiales W_q, W_k, W_v
    float* d_dX_q = W_q->backward(handle, d_Q_grad);
    float* d_dX_k = W_k->backward(handle, d_K_grad);
    float* d_dX_v = W_v->backward(handle, d_V_grad);

    // =====================================================================
    // ÉTAPE 7 : L'addition finale des gradients pour la couche précédente
    // =====================================================================
    int total_elements = batch_size * context_size * embedding_dim;
    int threads_sum = 256;
    int blocks_sum = (total_elements + threads_sum - 1) / threads_sum;

    // Lancement du kernel
    sum_gradients_kernel<<<blocks_sum, threads_sum>>>(
        d_dX,    // La destination (Le gradient final de la couche Attention)
        d_dX_q,  // Le gradient qui remonte de W_q
        d_dX_k,  // Le gradient qui remonte de W_k
        d_dX_v,  // Le gradient qui remonte de W_v
        total_elements
    );

    cudaDeviceSynchronize();
    return d_dX;
    }

void AttentionLayer::step(float learning_rate, int t) {
    W_q->step(learning_rate, t);
    W_k->step(learning_rate, t);
    W_v->step(learning_rate, t);
    W_o->step(learning_rate, t);
}


In [ ]:
%%writefile RMSNormLayer.cu
#include "RMSNormLayer.cuh"
#include <iostream>
#include <cmath>

// =========================================================================
// KERNELS CUDA
// =========================================================================

__global__ void rmsnorm_forward_kernel(float* d_X, float* d_Y, float* d_gamma, float* d_inv_rms, int embedding_dim, float epsilon) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    
    extern __shared__ float shared_sq_sum[];
    
    float val = d_X[row * embedding_dim + tid];
    shared_sq_sum[tid] = val * val;
    __syncthreads();
    
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            shared_sq_sum[tid] += shared_sq_sum[tid + stride];
        }
        __syncthreads();
    }
    
    if (tid == 0) {
        shared_sq_sum[0] = rsqrtf((shared_sq_sum[0] / embedding_dim) + epsilon); 
        d_inv_rms[row] = shared_sq_sum[0]; 
    }
    __syncthreads();
    
    float inv_rms = shared_sq_sum[0];
    d_Y[row * embedding_dim + tid] = val * inv_rms * d_gamma[tid];
}

// =========================================================================
// MÉTHODES DE LA CLASSE
// =========================================================================

// L'implémentation exacte qui correspond au .cuh
RMSNormLayer::RMSNormLayer(int batch_size, int context_size, int embedding_dim) 
    : Layer(batch_size, context_size, embedding_dim) {
    
    this->batch_size = batch_size;
    this->context_size = context_size;
    this->embedding_dim = embedding_dim;

    cudaMalloc(&d_gamma, sizeof(float) * embedding_dim);
    cudaMalloc(&d_inv_rms, sizeof(float) * batch_size * context_size);
    cudaMalloc(&d_dX, sizeof(float) * batch_size * context_size * embedding_dim);
    cudaMalloc(&d_Y, sizeof(float) * batch_size * context_size * embedding_dim);
    float* h_gamma = (float*)malloc(sizeof(float) * embedding_dim);
    for(int i = 0; i < embedding_dim; i++) h_gamma[i] = 1.0f;
    cudaMemcpy(d_gamma, h_gamma, sizeof(float) * embedding_dim, cudaMemcpyHostToDevice);
    free(h_gamma);
}

RMSNormLayer::~RMSNormLayer() {
    cudaFree(d_gamma);
    cudaFree(d_inv_rms);
    cudaFree(d_dX);
    cudaFree(d_Y);
}

float* RMSNormLayer::forward(cublasHandle_t handle, void* d_input, int activation_type) {
    float* d_X = (float*) d_input;
    this->d_X_cache = d_X;
    
    int total_words = batch_size * context_size;
    int threadsPerBlock = embedding_dim;
    size_t shared_mem = embedding_dim * sizeof(float);
    
    rmsnorm_forward_kernel<<<total_words, threadsPerBlock, shared_mem>>>(
        d_X, d_Y, d_gamma, d_inv_rms, embedding_dim, 1e-5f
    );
    
    cudaDeviceSynchronize();
    return d_Y;
}

// Le VRAI Kernel Backward de la RMSNorm (Mathématiquement exact)
__global__ void rmsnorm_backward_kernel(float* d_dY, float* d_X, float* d_dX, float* d_inv_rms, int embedding_dim) {
    int row = blockIdx.x; // 1 bloc = 1 mot
    int tid = threadIdx.x;

    extern __shared__ float shared_dot[];

    float dy = d_dY[row * embedding_dim + tid];
    float x  = d_X[row * embedding_dim + tid];

    // 1. On calcule le produit scalaire (dY * X)
    shared_dot[tid] = dy * x;
    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            shared_dot[tid] += shared_dot[tid + stride];
        }
        __syncthreads();
    }

    // 2. On applique la vraie formule de la dérivée
    float dot_sum = shared_dot[0];
    float inv = d_inv_rms[row];
    float scale = (inv * inv * inv) / embedding_dim;

    // La force de rappel magique est ce signe "moins" !
    d_dX[row * embedding_dim + tid] = (dy * inv) - (x * dot_sum * scale);
}

float* RMSNormLayer::backward(cublasHandle_t handle, float* d_dY) {
    int total_words = batch_size * context_size;
    int threadsPerBlock = embedding_dim;
    size_t shared_mem = embedding_dim * sizeof(float);
    
    // On appelle notre nouveau kernel ultra-robuste
    rmsnorm_backward_kernel<<<total_words, threadsPerBlock, shared_mem>>>(
        d_dY, d_X_cache, d_dX, d_inv_rms, embedding_dim
    );
    
    cudaDeviceSynchronize();
    return d_dX; 
}

void RMSNormLayer::step(float learning_rate, int t) {
    // Vide, Gamma reste figé
}


In [ ]:
%%writefile RMSNormLayer.cuh
#pragma once
#include "Layer.cuh"

class RMSNormLayer : public Layer {
  private:
    float* d_gamma;
    
    // Les tampons pour le backward bridé
    float* d_inv_rms; // Sauvegarde de la division (l'échelle)
    float* d_dX;      // Le gradient de sortie corrigé
    
    int batch_size;
    int context_size;
    int embedding_dim;

    float* d_Y;
    float* d_X_cache;

  public:
    // Le constructeur exact que le .cu va chercher
    RMSNormLayer(int batch_size, int context_size, int embedding_dim);
    ~RMSNormLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate, int t) override;
};


In [ ]:
%%writefile Layer.cuh
#pragma once // <-- INDISPENSABLE pour éviter la redéfinition !
#include <cublas_v2.h>

// Définitions de nos constantes d'activation
#define ACTIVATION_NONE 0
#define ACTIVATION_RELU 1
#define ACTIVATION_GELU 2

class Layer {
  protected:
    int batch_features;
    int in_features;
    int out_features;

  public:
    // Constructeur de base
    Layer(int batch_size, int in_feat, int out_feat) {
        this->batch_features = batch_size;
        this->in_features = in_feat;
        this->out_features = out_feat;
    }

    // Destructeur virtuel obligatoire quand on fait de l'héritage
    virtual ~Layer() {}

    // LE CONTRAT (Pure virtual functions)
    // ATTENTION : On utilise bien un void* pour d_input ici !
    virtual float* forward(cublasHandle_t handle, void* d_input, int activation_type) = 0; 
    virtual float* backward(cublasHandle_t handle, float* d_dY) = 0;
    virtual void step(float learning_rate,int t) = 0;
};


In [ ]:
%%writefile AttentionLayer.cuh
#pragma once
#include "Layer.cuh"
#include "LinearLayer.cuh" // On importe notre propre outil !

class AttentionLayer : public Layer {
  private:
    // 1. Nos sous-couches (Les générateurs de Q, K et V)
    LinearLayer* W_q;
    LinearLayer* W_k;
    LinearLayer* W_v;
    
    // (Optionnel mais standard) Une dernière couche linéaire pour mélanger 
    // le résultat final avant de le passer au MLP
    LinearLayer* W_o; 

    // 2. Nos espaces mémoires intermédiaires sur le GPU
    float* d_Q;
    float* d_K;
    float* d_V;
    float* d_Scores;       // Pour stocker (Q * K^T)
    float* d_AttentionOut; // Pour stocker (Scores * V)
    
    // 3. Les dimensions
    int batch_size;
    int context_size;
    int embedding_dim;
    // NOUVEAU : Les tampons (buffers) pour le Backward !
    float* d_Q_grad;
    float* d_K_grad;
    float* d_V_grad;
    float* d_dScores_softmax;
    float* d_dScores;
    float* d_dX; // Le gradient final qui ressort de l'Attention

  public:
    AttentionLayer(int batch_size, int context_size, int embedding_dim);
    ~AttentionLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile FeedForwardLayer.cuh
#pragma once
#include "Layer.cuh"
#include "LinearLayer.cuh"

class FeedForwardLayer : public Layer {
  private:
    // Nos deux sous-couches
    LinearLayer* fc1; // L'expansion (x4)
    LinearLayer* fc2; // La contraction (x1)

    // Dimensions
    int batch_size;
    int context_size;
    int embedding_dim;
    int hidden_dim; // Généralement 4 * embedding_dim

  public:
    FeedForwardLayer(int batch_size, int context_size, int embedding_dim, int expansion_factor = 4);
    ~FeedForwardLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile EmbeddingLayer.cuh
#pragma once
#include "Layer.cuh"

class EmbeddingLayer : public Layer {
  private:
    // Poids et gradients
    float* d_W;  // Le dictionnaire [vocab_size * embedding_dim]
    float* d_dW; // Le gradient des poids

    // Encodage Positionnel (Constant)
    float* d_PE; // La matrice des ondes [context_size * embedding_dim]
    
    // Entrées / Sorties
    int* d_X;    // L'entrée (Tableau d'entiers) [batch_size * context_size]
    float* d_Y;  // La sortie [batch_size * context_size * embedding_dim]
    
    // Les dimensions
    int vocab_size;
    int embedding_dim;
    int batch_size;
    int context_size;

    //ADAM
    float* d_m;
    float* d_v;
    
  public:
    EmbeddingLayer(int vocab_size, int embedding_dim, int batch_size, int context_size);
    ~EmbeddingLayer();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile LinearLayer.cuh
#pragma once
#include "Layer.cuh"
#include <cublas_v2.h>

class LinearLayer : public Layer {
  private:
    float* d_W;
    float* d_b;
    float* d_dW;
    float* d_db;
    
    float* d_X_cache; // Pour sauvegarder l'entrée pendant le forward
    float* d_Y;       // La sortie
    float* d_dX;      // Le gradient à renvoyer vers le bas

    // ADAM
    float* d_m; // Momentum
    float* d_v; // Vélocité

  public:
    LinearLayer(int batch_size, int in_feat, int out_feat);
    ~LinearLayer();

    // Les 3 fameuses méthodes imposées par Layer.cuh
    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile GPTModel.cuh
#pragma once
#include "Layer.cuh"
#include "EmbeddingLayer.cuh"
#include "TransformerBlock.cuh"
#include "RMSNormLayer.cuh"
#include "LinearLayer.cuh"
#include <vector>

class GPTModel : public Layer {
  private:
    // La liste complète de nos composants
    EmbeddingLayer* embedding;
    
    // Un tableau dynamique pour stocker nos N blocs Transformer
    std::vector<TransformerBlock*> blocks; 
    
    RMSNormLayer* final_norm;
    LinearLayer* lm_head; // La tête de prédiction

    // L'architecture
    int vocab_size;
    int embedding_dim;
    int batch_size;
    int context_size;
    int num_blocks; // Combien de couches d'Attention on empile ?

  public:
    GPTModel(int vocab_size, int embedding_dim, int batch_size, int context_size, int num_blocks);
    ~GPTModel();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override; 
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate,int t) override;
};


In [ ]:
%%writefile TransformerBlock.cuh
#pragma once
#include "Layer.cuh"
#include "AttentionLayer.cuh"
#include "FeedForwardLayer.cuh"
#include "RMSNormLayer.cuh"

class TransformerBlock : public Layer {
private:
    RMSNormLayer* norm1;
    AttentionLayer* attn;
    RMSNormLayer* norm2;
    FeedForwardLayer* ffn;

    // NOUVEAU : Mémoire pour les câbles de contournement (Residuals)
    float* d_res1; // Stocke : Entrée + Attention
    float* d_out;  // Stocke : res1 + FeedForward

    // NOUVEAU : Mémoire pour remonter les gradients
    float* d_dRes1; 
    float* d_dX;    // Le gradient final à renvoyer en dessous

    int batch_size;
    int context_size;
    int embedding_dim;

public:
    TransformerBlock(int batch_size, int context_size, int embedding_dim);
    ~TransformerBlock();

    float* forward(cublasHandle_t handle, void* d_input, int activation_type) override;
    float* backward(cublasHandle_t handle, float* d_dY) override;
    void step(float learning_rate, int t) override;
};


In [ ]:
%%writefile DataLoader.cpp
#include "DataLoader.h"
#include <fstream>
#include <sstream>
#include <iostream>
#include <set>
#include <cstdlib>

DataLoader::DataLoader(const std::string& filepath, int batch_size, int context_size) {
    this->batch_size = batch_size;
    this->context_size = context_size;

    // 1. Lecture du fichier texte
    std::ifstream file(filepath);
    if (!file.is_open()) {
        std::cerr << "ERREUR CRITIQUE : Impossible d'ouvrir le fichier " << filepath << std::endl;
        exit(1);
    }
    std::stringstream buffer;
    buffer << file.rdbuf();
    raw_text = buffer.str();
    file.close();

    // 2. Création du vocabulaire (Trouver les caractères uniques)
    std::set<char> unique_chars(raw_text.begin(), raw_text.end());
    vocab_size = unique_chars.size();

    // 3. Remplissage des dictionnaires
    int i = 0;
    for (char c : unique_chars) {
        char_to_int[c] = i;
        int_to_char[i] = c;
        i++;
    }

    // 4. Tokenization : On convertit tout le texte en chiffres !
    tokens.reserve(raw_text.size());
    for (char c : raw_text) {
        tokens.push_back(char_to_int[c]);
    }

    std::cout << "DataLoader initialise ! Taille du texte: " << tokens.size() 
              << " caracteres, Vocabulaire: " << vocab_size << " caracteres." << std::endl;
}

DataLoader::~DataLoader() {}

void DataLoader::get_batch(int* h_X, int* h_targets) {
    // Pour chaque ligne du batch
    for (int b = 0; b < batch_size; b++) {
        // On tire un index de départ au hasard (en laissant la place pour le context_size + la cible)
        int max_start_idx = tokens.size() - context_size - 1;
        int start_idx = rand() % max_start_idx;

        // On remplit le contexte (X) et la cible décalée d'un cran (Y)
        for (int i = 0; i < context_size; i++) {
            h_X[b * context_size + i] = tokens[start_idx + i];
            h_targets[b * context_size + i] = tokens[start_idx + i + 1];
        }
    }
}


In [ ]:
%%writefile DataLoader.h
#pragma once
#include <string>
#include <vector>
#include <map>

class DataLoader {
private:
    std::string raw_text;
    std::vector<int> tokens; // Le texte entier converti en nombres
    
    // Nos dictionnaires de traduction
    std::map<char, int> char_to_int;
    std::map<int, char> int_to_char;
    
    int batch_size;
    int context_size;
    int vocab_size;

public:
    DataLoader(const std::string& filepath, int batch_size, int context_size);
    ~DataLoader();

    // Remplit les tableaux CPU avec un nouveau batch tiré au hasard
    void get_batch(int* h_X, int* h_targets);
    
    // Getters utiles pour configurer le modèle
    int get_vocab_size() const { return vocab_size; }
    std::map<int, char> get_int_to_char_map() const { return int_to_char; }
};


In [ ]:
# Téléchargement du jeu de données Tiny Shakespeare
!wget -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
!nvcc -O3 -o mon_gpt main.cu GPTModel.cu TransformerBlock.cu AttentionLayer.cu FeedForwardLayer.cu RMSNormLayer.cu EmbeddingLayer.cu LinearLayer.cu DataLoader.cpp -lcublas
!./mon_gpt
